In [99]:
%%capture
from pathlib import Path

if Path.cwd().stem == "notebooks":
    %cd ..
    %load_ext autoreload
    %autoreload 2

In [100]:
import altair as alt
import holoviews as hv
import hvplot.polars  # noqa
import numpy as np
import polars as pl
from polars import col

from src.data.database_manager import DatabaseManager
from src.features.resampling import add_normalized_timestamp, add_timestamp_μs_column
from src.log_config import configure_logging
from src.plots.averages_over_stimulus_seeds import (
    average_over_stimulus_seeds,
    calculate_crosscorr_matrix,
    plot_averages_with_ci_plt,
    plot_correlation_heatmap,
)
from src.plots.utils import prepare_multiline_hvplot

configure_logging(
    ignore_libs=("Comm", "bokeh", "tornado", "matplotlib"),
)

pl.Config.set_tbl_rows(12)  # for 12 seeds
hv.extension("bokeh")
hv.output(widget_location="bottom", size=150)

In [101]:
db = DatabaseManager()

In [102]:
with db:
    df = db.get_trials("Explore_Data", exclude_problematic=True)
df

trial_id,trial_number,participant_id,timestamp,temperature,rating,eda_raw,eda_tonic,eda_phasic,eda_tonic_detrended,ppg_raw,heart_rate,ibi,pupil_l_raw,pupil_r_raw,pupil_r,pupil_l,pupil,brow_furrow,cheek_raise,mouth_open,upper_lip_raise,nose_wrinkle,normalized_timestamp,stimulus_seed,skin_patch,decreasing_intervals,major_decreasing_intervals,increasing_intervals,strictly_increasing_intervals,strictly_increasing_intervals_without_plateaus,plateau_intervals,prolonged_minima_intervals
u16,u8,u8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u16,u8,u16,u16,u16,u16,u16,u16,u16
1,1,1,200180.9348,0.0,0.38375,19.652376,19.436877,0.215499,1.389781,1372.496197,75.75827,3.021361,4.784992,4.444289,4.450926,4.785598,4.618262,0.205826,0.00012,0.00039,0.000317,0.000665,0.0,870,1,0,0,1,1,1,0,0
1,1,1,200280.9348,0.000056,0.395091,19.793001,19.453226,0.339775,1.405727,1365.584792,75.95712,-4.283564,4.82506,4.452629,4.449446,4.789018,4.619232,0.192546,0.00012,0.000411,0.000309,0.000642,100.0,870,1,0,0,1,1,1,0,0
1,1,1,200380.9348,0.000241,0.406316,19.884253,19.466441,0.417812,1.418614,1356.423933,76.15928,-0.763131,4.835159,4.469986,4.446379,4.789661,4.61802,0.1817,0.000119,0.000432,0.000303,0.000625,200.0,870,1,0,0,1,1,1,0,0
1,1,1,200480.9348,0.000572,0.415328,20.012645,19.486828,0.525816,1.438493,1397.727216,76.585482,1.514833,4.842851,4.481221,4.449453,4.788254,4.618853,0.175569,0.000118,0.00045,0.000302,0.00062,300.0,870,1,0,0,1,1,1,0,0
1,1,1,200580.9348,0.001049,0.424044,20.094258,19.50245,0.591808,1.453722,1444.547187,77.051665,-12.188997,4.829974,4.489243,4.444739,4.789912,4.617325,0.172782,0.000117,0.00046,0.000302,0.000618,400.0,870,1,0,0,1,1,1,0,0
1,1,1,200680.9348,0.001676,0.435615,20.156349,19.518017,0.638333,1.468894,1419.597873,77.653838,46.059466,4.798762,4.477897,4.452002,4.787977,4.61999,0.170531,0.000115,0.00048,0.000304,0.000624,500.0,870,1,0,0,1,1,1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
496,4,42,1.1405e6,0.031406,0.0,15.030559,15.008543,0.022016,-2.603602,1434.106413,62.353678,-1.19803,3.577106,3.476652,3.50111,3.600631,3.550871,0.015639,0.000177,0.139924,0.002577,0.002907,179500.0,806,3,2395,1437,0,0,0,0,0
496,4,42,1.1406e6,0.030945,0.0,15.025569,15.006031,0.019538,-2.606329,1412.519621,62.199718,-0.764415,3.548026,3.485169,3.49922,3.594389,3.546804,0.016379,0.000181,0.145722,0.002668,0.002891,179600.0,806,3,2395,1437,0,0,0,0,0


In [103]:
corr_by_trial = (
    df.group_by("trial_id", "trial_number")
    .agg(pl.corr("rating", "temperature").alias("correlation"))
    .sort("trial_number")
)
corr_by_trial

trial_id,trial_number,correlation
u16,u8,f64
241,1,0.803059
109,1,0.72836
205,1,0.592997
61,1,0.423785
181,1,0.605471
481,1,0.684669
…,…,…
444,12,0.74512
408,12,0.459609


In [104]:
from scipy import stats

groups = [
    group["correlation"].to_numpy()
    for _, group in corr_by_trial.group_by("trial_number", maintain_order=True)
]

f_stat, p_value_anova = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.3f}, p-value: {p_value_anova:.3f}")

F-statistic: 1.704, p-value: 0.070


In [105]:
slope, intercept, r_value, p_value, std_err = stats.linregress(
    corr_by_trial["trial_number"],
    corr_by_trial["correlation"],
)
print(f"Slope: {slope:.4f}, p-value: {p_value:.3f}, R²: {r_value**2:.3f}")

Slope: 0.0023, p-value: 0.032, R²: 0.010


In [106]:
alt.Chart(corr_by_trial).mark_boxplot().encode(
    x=alt.X("trial_number:O", axis=alt.Axis(labelAngle=0), title="Trial Number"),
    y=alt.Y("correlation:Q", title="Correlation Between Pain Rating and Temperature"),
).properties(
    width=600,
    height=300,
)

alt.Chart(...)

In [107]:
major_decrease_extrema = (
    df.filter(
        (col("major_decreasing_intervals") > 0)
        & col("rating").is_not_null()
        & col("normalized_timestamp").is_not_null()
    )
    .group_by(["trial_id", "trial_number", "major_decreasing_intervals"])
    .agg(
        col("rating").sort_by("normalized_timestamp").max().alias("max_rating"),
        col("rating").sort_by("normalized_timestamp").min().alias("min_rating"),
    )
)

extrema_ratings = major_decrease_extrema.unpivot(
    index=["trial_id", "trial_number", "major_decreasing_intervals"],
    on=["max_rating", "min_rating"],
    variable_name="rating_point",
    value_name="rating",
)


In [108]:
boxplot = (
    alt.Chart(extrema_ratings)
    .mark_boxplot(size=30, outliers={"size": 15, "opacity": 0.3})
    .encode(
        x=alt.X("trial_number:O", title="Trial Number"),
        y=alt.Y("rating:Q", title="Pain Rating (VAS)"),
        color=alt.Color("rating_point:N"),
        xOffset=alt.XOffset(
            "rating_point:N",
            scale=alt.Scale(paddingInner=1, paddingOuter=1),  # tighten spacing
        ),
    )
)

boxplot.properties(
    width=600,
    height=300,
    # title="Max vs min pain ratings in major decreases across trial numbers",
).configure_axis(grid=False)

alt.Chart(...)

In [109]:
decreases = df.filter(col("major_decreasing_intervals") > 0)
decreases = decreases.drop("timestamp")
decreases = decreases.rename({"normalized_timestamp": "timestamp"})

In [110]:
def add_normalized_timestamp(
    df: pl.DataFrame,
    time_column: str = "timestamp",
    trial_column: str = "trial_id",
):
    return df.with_columns(
        [
            (col(time_column) - col(time_column).min().over(trial_column))
            .alias("normalized_timestamp")
            .cast(pl.Int16)
        ]
    )


In [111]:
decreases = add_normalized_timestamp(
    decreases, time_column="timestamp", trial_column="major_decreasing_intervals"
)

In [112]:
decreases = decreases.drop(
    [
        "timestamp",
        "eda_raw",
        "eda_tonic",
        "eda_phasic",
        "eda_tonic_detrended",
        "ppg_raw",
        "heart_rate",
        "ibi",
        "pupil_l_raw",
        "pupil_r_raw",
        "pupil_r",
        "pupil_l",
        "pupil",
        "brow_furrow",
        "cheek_raise",
        "mouth_open",
        "upper_lip_raise",
        "nose_wrinkle",
        "stimulus_seed",
        "skin_patch",
        "decreasing_intervals",
        "increasing_intervals",
        "strictly_increasing_intervals",
        "strictly_increasing_intervals_without_plateaus",
        "plateau_intervals",
        "prolonged_minima_intervals",
    ]
)


In [113]:
def ci_layer(source, color, mean_col, lower_col, upper_col, label, show_y_title=True):
    source = source.clone().with_columns(
        [
            pl.lit(label).alias("label"),
            (pl.col("normalized_timestamp") / 1000).alias("normalized_timestamp"),
            (pl.col(mean_col) * 70).alias(mean_col),
            (pl.col(lower_col) * 70).alias(lower_col),
            (pl.col(upper_col) * 70).alias(upper_col),
        ]
    )
    color_scale = alt.Color(
        "label:N",
        scale=alt.Scale(domain=["Temperature", "Rating"], range=["steelblue", "coral"]),
        legend=alt.Legend(
            title="",
            orient="none",
            legendX=10,
            legendY=10,
            direction="horizontal",
        ),
    )

    base = alt.Chart(source).encode(x=alt.X("normalized_timestamp:Q", title="Time (s)"))

    line = base.mark_line().encode(
        y=alt.Y(
            f"{mean_col}:Q",
            title="Pain Rating (VAS)" if show_y_title else None,
            axis=alt.Axis() if show_y_title else None,
        ),
        color=color_scale,
        tooltip=["normalized_timestamp:Q", f"{mean_col}:Q"],
    )

    band = base.mark_area(opacity=0.2).encode(
        y=alt.Y(f"{lower_col}:Q", axis=alt.Axis() if show_y_title else None),
        y2=alt.Y2(f"{upper_col}:Q"),
        color=color_scale,
    )

    return line + band


temp_layer = ci_layer(
    ci_values,
    "steelblue",
    "mean_temperature",
    "ci_lower_temperature",
    "ci_upper_temperature",
    "Temperature",
)
rating_layer = ci_layer(
    ci_values,
    "coral",
    "mean_rating",
    "ci_lower_rating",
    "ci_upper_rating",
    "Rating",
    show_y_title=False,
)

background = (
    alt.Chart(
        alt.Data(values=[{"x1": 1, "x2": 8}])  # updated from 1000/8000 to 1/8
    )
    .mark_rect(opacity=0.1, color="gray")
    .encode(
        x=alt.X("x1:Q"),
        x2=alt.X2("x2:Q"),
    )
)

(background + temp_layer + rating_layer).resolve_scale(y="independent").properties(
    width=600,
    height=300,
)

NameError: name 'ci_values' is not defined